# Blackstart City — Agent Tier Demo

Shows the **3-tier cascade** working on real blackout scenarios:

| Tier | Policy | Reasoning |
|------|--------|----------|
| 0 | **GreedyPolicy** | Fast, no reasoning — often violates constraints |
| 1 | **HeuristicPolicy** | Shortest-path planning — better, but news-blind |
| 2 | **LLMPolicy (GRPO)** | Sees ALL prior failures — Theory-of-Mind reasoning |

On each failure the failure context is injected into the next tier's observations.
The LLM (Tier 2) is the GRPO-trained model loaded from HuggingFace.

**Runtime:** L4 / A100 recommended (GPU needed for Tier 2 inference).

In [ ]:
# ── 1. Install ────────────────────────────────────────────────────────────────
!pip install unsloth peft transformers accelerate huggingface_hub -q
!pip install git+https://github.com/Ankitpatil944/blackout-city.git -q

In [ ]:
# ── 2. Download GRPO adapter from HF Space ────────────────────────────────────
import os, shutil
from huggingface_hub import snapshot_download

HF_SPACE   = "SidditaVarma/Built-different"
GRPO_DIR   = "grpo_adapter"
SFT_DIR    = "sft_adapter"

if not os.path.isdir(GRPO_DIR):
    snapshot_download(
        repo_id        = HF_SPACE,
        repo_type      = "space",
        allow_patterns = ["latest/blackstart-city-grpo/**"],
        local_dir      = ".",
    )
    shutil.move("latest/blackstart-city-grpo", GRPO_DIR)
    print(f"Downloaded GRPO adapter → {GRPO_DIR}/")
else:
    print(f"Reusing {GRPO_DIR}/")

if not os.path.isdir(SFT_DIR):
    snapshot_download(
        repo_id        = HF_SPACE,
        repo_type      = "space",
        allow_patterns = ["latest/blackstart-city-sft/**"],
        local_dir      = ".",
    )
    shutil.move("latest/blackstart-city-sft", SFT_DIR)
    print(f"Downloaded SFT adapter  → {SFT_DIR}/")
else:
    print(f"Reusing {SFT_DIR}/")

import json
cfg = json.load(open(f"{GRPO_DIR}/adapter_config.json"))
BASE_MODEL = cfg["base_model_name_or_path"]
print(f"Base model: {BASE_MODEL}")

In [ ]:
# ── 3. Load GRPO model (used by Tier 2) ──────────────────────────────────────
import torch
from unsloth import FastLanguageModel

grpo_model, grpo_tokenizer = FastLanguageModel.from_pretrained(
    model_name             = BASE_MODEL,
    max_seq_length         = 4096,
    load_in_4bit           = True,
    fast_inference         = False,
)
grpo_model.load_adapter(GRPO_DIR, adapter_name="grpo", is_trainable=False)
grpo_model.set_adapter("grpo")
FastLanguageModel.for_inference(grpo_model)
print("GRPO model ready.")

# Also load SFT model for comparison
sft_model, sft_tokenizer = FastLanguageModel.from_pretrained(
    model_name             = BASE_MODEL,
    max_seq_length         = 4096,
    load_in_4bit           = True,
    fast_inference         = False,
)
sft_model.load_adapter(SFT_DIR, adapter_name="sft", is_trainable=False)
sft_model.set_adapter("sft")
FastLanguageModel.for_inference(sft_model)
print("SFT model ready.")

In [ ]:
# ── 4. Wire LLMPolicy to the real GRPO model ──────────────────────────────────
import json
import torch
from typing import Any, Dict
from blackstart_city.models import BlackstartAction, ActionType
from blackstart_city.training.model_utils import build_policy_prompt, parse_action_text
from blackstart_city.agent_tier import (
    AgentTier, GreedyPolicy, HeuristicPolicy, LLMPolicy,
    _to_observation, _fallback_status
)

def _llm_act(model, tokenizer, observation: Dict[str, Any]) -> BlackstartAction:
    """Run the loaded model on an observation dict and return a parsed action."""
    obs = _to_observation(observation)
    prompt = build_policy_prompt(obs)

    # Add failure context to prompt if present
    failure_ctx = observation.get("failure_context", [])
    if failure_ctx:
        ctx_str = json.dumps(failure_ctx, indent=2)
        prompt = prompt + f"\n\n[Prior tier failures]\n{ctx_str}"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=150,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)
    action = parse_action_text(text)
    if action is None:
        action = BlackstartAction(
            action_type=ActionType.PUBLISH_STATUS,
            status_update=_fallback_status()
        )
    return action, text


# Patch LLMPolicy to use the real GRPO model
class GRPOLLMPolicy:
    name = "LLMPolicy (GRPO)"
    def __init__(self):
        self._published = False
        self._seen_signatures: set = set()
    def act(self, observation: Dict[str, Any]) -> BlackstartAction:
        action, _ = _llm_act(grpo_model, grpo_tokenizer, observation)
        return action

# Patch AgentTier to use GRPOLLMPolicy as Tier 2
AgentTier.TIER_CLASSES = [GreedyPolicy, HeuristicPolicy, GRPOLLMPolicy]
AgentTier.TIER_NAMES   = ["GreedyPolicy", "HeuristicPolicy", "LLMPolicy (GRPO)"]

print("AgentTier wired to GRPO model.")
print("Tiers:", AgentTier.TIER_NAMES)

In [ ]:
# ── 5. Run all 3 scenarios through the full tier cascade ──────────────────────
from blackstart_city.env import BlackstartCityEnv

SCENARIOS = [
    ("local_blackstart",      "EASY",   "Restart dark district, restore hospital"),
    ("city_cascade_recovery", "HARD",   "Recover city while critical services degrade"),
    ("mega_cascade",          "HARD",   "Two hospitals, conflicting orders, 8-min backup"),
]

all_results = {}

for task_id, difficulty, description in SCENARIOS:
    print(f"\n{'='*60}")
    print(f"Scenario : {task_id}")
    print(f"Difficulty: {difficulty}")
    print(f"Goal     : {description}")
    print(f"{'='*60}")

    env = BlackstartCityEnv()
    agent_tier = AgentTier()
    result = agent_tier.run(env, task_id=task_id, seed=42)

    print(f"\nResult   : {result}")
    print(f"Tier used: {result.tier_name} (tier {result.tier_used})")
    print(f"Score    : {result.score:.3f}")
    print(f"Time     : {result.wall_seconds:.1f}s")

    if result.failure_contexts:
        print(f"\nEscalation chain:")
        for fc in result.failure_contexts:
            print(f"  Tier {fc['failed_tier']} ({fc['failed_tier_name']}) failed "
                  f"| score={fc['score_at_failure']:.3f} | reason: {fc['failure_reason'][:80]}")

    all_results[task_id] = result

In [ ]:
# ── 6. Side-by-side: SFT vs GRPO on the same hard scenario ───────────────────
from blackstart_city.env import BlackstartCityEnv

TASK = "city_cascade_recovery"
SEED = 42

print(f"Comparing SFT vs GRPO on '{TASK}' (seed={SEED})\n")

results_comparison = {}
for label, model, tokenizer in [
    ("SFT",  sft_model,  sft_tokenizer),
    ("GRPO", grpo_model, grpo_tokenizer),
]:
    env = BlackstartCityEnv()
    obs = env.reset(task_id=TASK, seed=SEED)

    done = False
    actions_taken = []
    raw_outputs   = []
    total_reward  = 0.0
    published     = False
    seen_sigs: set = set()

    while not done:
        obs_dict = obs if isinstance(obs, dict) else obs.model_dump()
        action, raw_text = _llm_act(model, tokenizer, obs_dict)
        raw_outputs.append(raw_text.strip()[:120])
        obs, reward, done, info = env.step(action)
        total_reward += reward
        a = action if isinstance(action, dict) else action.model_dump(exclude_none=True)
        actions_taken.append(a)

    score = float(info.get("score", total_reward))
    success = bool(info.get("success", score >= 0.6))
    results_comparison[label] = {
        "score": score,
        "success": success,
        "steps": len(actions_taken),
        "actions": actions_taken,
        "outputs": raw_outputs,
    }

    print(f"{'[' + label + ']':8s}  score={score:.3f}  success={success}  steps={len(actions_taken)}")

print("\n── First 3 actions side by side ──────────────────────────────────────")
for i in range(min(3, len(results_comparison['SFT']['actions']))):
    sft_a  = results_comparison['SFT']['actions'][i]
    grpo_a = results_comparison['GRPO']['actions'][i]
    print(f"\nStep {i+1}:")
    print(f"  SFT : {sft_a}")
    print(f"  GRPO: {grpo_a}")

In [ ]:
# ── 7. Visualisations ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.style.use("dark_background")
fig, axes = plt.subplots(1, 3, figsize=(22, 7), facecolor="#121212")
fig.suptitle("Blackstart City — Agent Tier Results", color="#00FFCC", fontsize=15, fontweight="bold")

# ── Panel 1: Tier cascade per scenario ───────────────────────────────────────
ax = axes[0]
ax.set_facecolor("#1e1e1e")
scenario_labels = [s[0].replace("_", "\n") for s in SCENARIOS]
tier_colors = ["#FF6B6B", "#FFD700", "#00FF66"]
tier_labels_legend = ["Tier 0: Greedy", "Tier 1: Heuristic", "Tier 2: LLM (GRPO)"]

for i, (task_id, _, _) in enumerate(SCENARIOS):
    r = all_results[task_id]
    color = tier_colors[r.tier_used]
    bar = ax.bar(i, r.score, color=color, alpha=0.85, width=0.6)
    ax.text(i, r.score + 0.02, f"{r.score:.2f}", ha="center", va="bottom",
            color="white", fontsize=9, fontweight="bold")
    status = "SUCCESS" if r.success else "FAILED"
    ax.text(i, -0.08, status, ha="center", va="top",
            color="#00FF66" if r.success else "#FF6B6B", fontsize=8)

ax.set_xticks(range(len(SCENARIOS)))
ax.set_xticklabels(scenario_labels, fontsize=8)
ax.set_ylim(-0.15, 1.15)
ax.set_ylabel("Score")
ax.set_title("Score by Scenario\n(colour = tier that solved it)", color="#00FFCC", fontweight="bold")
patches = [mpatches.Patch(color=c, label=l) for c, l in zip(tier_colors, tier_labels_legend)]
ax.legend(handles=patches, fontsize=7, facecolor="#1e1e1e", edgecolor="#444", labelcolor="white")
ax.grid(True, linestyle="--", alpha=0.15, axis="y")

# ── Panel 2: Escalation waterfall for city_cascade_recovery ──────────────────
ax2 = axes[1]
ax2.set_facecolor("#1e1e1e")
hard_result = all_results["city_cascade_recovery"]
all_tiers = [fc["failed_tier_name"] for fc in hard_result.failure_contexts] + [hard_result.tier_name]
all_scores = [fc["score_at_failure"] for fc in hard_result.failure_contexts] + [hard_result.score]
colors_w = [tier_colors[fc["failed_tier"]] for fc in hard_result.failure_contexts] + [tier_colors[hard_result.tier_used]]

bars = ax2.bar(range(len(all_tiers)), all_scores, color=colors_w, alpha=0.85, width=0.6)
for i, (score, name) in enumerate(zip(all_scores, all_tiers)):
    ax2.text(i, score + 0.02, f"{score:.2f}", ha="center", va="bottom",
             color="white", fontsize=9, fontweight="bold")
    failed = i < len(hard_result.failure_contexts)
    ax2.text(i, -0.08, "FAIL" if failed else "SOLVED",
             ha="center", va="top",
             color="#FF6B6B" if failed else "#00FF66", fontsize=8)

ax2.set_xticks(range(len(all_tiers)))
ax2.set_xticklabels([t.replace(" ", "\n") for t in all_tiers], fontsize=8)
ax2.set_ylim(-0.15, 1.15)
ax2.set_ylabel("Score at Handoff")
ax2.set_title("Escalation Waterfall\n(city_cascade_recovery)", color="#00FFCC", fontweight="bold")
ax2.grid(True, linestyle="--", alpha=0.15, axis="y")

# ── Panel 3: SFT vs GRPO score comparison ────────────────────────────────────
ax3 = axes[2]
ax3.set_facecolor("#1e1e1e")
models_cmp = ["SFT", "GRPO"]
scores_cmp = [results_comparison[m]["score"] for m in models_cmp]
colors_cmp = ["#FFD700", "#00FF66"]

bars3 = ax3.bar(models_cmp, scores_cmp, color=colors_cmp, alpha=0.85, width=0.4)
for i, (m, s) in enumerate(zip(models_cmp, scores_cmp)):
    ax3.text(i, s + 0.02, f"{s:.3f}", ha="center", va="bottom",
             color="white", fontsize=12, fontweight="bold")
    status = results_comparison[m]["success"]
    ax3.text(i, -0.08, "SUCCESS" if status else "FAILED",
             ha="center", va="top",
             color="#00FF66" if status else "#FF6B6B", fontsize=9)

if scores_cmp[1] > scores_cmp[0]:
    delta = scores_cmp[1] - scores_cmp[0]
    ax3.annotate(f"+{delta:.3f}",
                 xy=(1, scores_cmp[1]), xytext=(0.5, max(scores_cmp) + 0.12),
                 color="#00FFCC", fontsize=11, fontweight="bold",
                 arrowprops=dict(arrowstyle="->", color="#00FFCC"),
                 ha="center")

ax3.set_ylim(-0.15, 1.25)
ax3.set_ylabel("Score")
ax3.set_title(f"SFT vs GRPO\n({TASK})", color="#00FFCC", fontweight="bold")
ax3.grid(True, linestyle="--", alpha=0.15, axis="y")

plt.tight_layout()
os.makedirs("demo_output", exist_ok=True)
plt.savefig("demo_output/agent_tier_demo.png", dpi=150, bbox_inches="tight", facecolor="#121212")
plt.show()
print("Saved → demo_output/agent_tier_demo.png")

In [ ]:
# ── 8. Summary table ──────────────────────────────────────────────────────────
print("\n" + "="*70)
print(f"{'Scenario':<28} {'Tier Solved':<22} {'Score':>7} {'Status':>8}")
print("-"*70)
for task_id, difficulty, _ in SCENARIOS:
    r = all_results[task_id]
    status = "SUCCESS" if r.success else "FAILED"
    print(f"{task_id:<28} {r.tier_name:<22} {r.score:>7.3f} {status:>8}")

print("="*70)
print(f"\nSFT  score on {TASK}: {results_comparison['SFT']['score']:.3f}  "
      f"({'SUCCESS' if results_comparison['SFT']['success'] else 'FAILED'})")
print(f"GRPO score on {TASK}: {results_comparison['GRPO']['score']:.3f}  "
      f"({'SUCCESS' if results_comparison['GRPO']['success'] else 'FAILED'})")
delta = results_comparison['GRPO']['score'] - results_comparison['SFT']['score']
print(f"GRPO improvement   : {delta:+.3f}")

In [ ]:
# ── 9. (Optional) Upload demo graph to HF Space ──────────────────────────────
from huggingface_hub import HfApi, login

HF_TOKEN = ""   # paste your write token

if HF_TOKEN:
    login(token=HF_TOKEN)
    api = HfApi()
    api.upload_file(
        path_or_fileobj = "demo_output/agent_tier_demo.png",
        path_in_repo    = "latest/blackstart-city-grpo/agent_tier_demo.png",
        repo_id         = "SidditaVarma/Built-different",
        repo_type       = "space",
        commit_message  = "Add agent tier demo graph",
    )
    print("Uploaded agent_tier_demo.png to HF Space.")
else:
    print("Set HF_TOKEN to upload.")